In [1]:
from functions.model import ContourCNNSelector
from functions.model_interface import run_leave_instance_out, run_leave_problem_out, run_random_split
import pandas as pd

df = pd.read_csv("data/relert_bbob.csv")

alg_cols = df.columns[2:].tolist()
make_model = lambda: ContourCNNSelector(num_algorithms=len(alg_cols))

resolution = 16
k_views = 16

# Protocols:
# - 5-fold instance CV (true 5-fold partitioning over instances): run_5fold_instance_cv
# - Leave-instance-out (LIO): run_leave_instance_out

train_func = run_leave_instance_out

preds_by_fold = train_func(
    df,
    data_root="data/bbob",          # folder containing ./bbob/res_*
    resolution=resolution,
    k_views=k_views,
    num_repetitions=10,
    make_model=make_model,
    batch_size=32,
    num_epochs=50,
    lr=3e-4,
    cache_train=True,
    cache_test=True,
 )

# How to save a dictionary of dataframes 
import pickle
with open("preds_by_fold.pkl", "wb") as f:
    pickle.dump(preds_by_fold, f)

/data1/home/jw1017/miniforge3/envs/as_bbo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[train LIO 5/5] epoch: 100%|██████████| 50/50 [00:52<00:00,  1.05s/epoch, loss=1.075857]


In [ ]:
# How to load it back
import pickle
with open("preds_by_fold.pkl", "rb") as f:
    preds_by_fold = pickle.load(f)

from functions.model_interface import output_results
import pandas as pd

df = pd.read_csv("data/relert_bbob.csv")
# df = pd.read_csv("data/log_relert_bbob.csv")

metrics = output_results(df, preds_by_fold)

# Save all summary tables into ONE CSV under results/bbob/
import os
from datetime import datetime

out_dir = "results/bbob/LPO"
os.makedirs(out_dir, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(out_dir, f"res_{resolution}_k_views_{k_views}.csv")

with open(out_path, "w") as f:
    f.write("Mean\n")
    f.write("AS\n")
    metrics["scores"].to_csv(f, index=False)
    f.write("\nVBS\n")
    metrics["vbs"].to_csv(f, index=False)
    f.write("\nSBS\n")
    metrics["sbs"].to_csv(f, index=False)
    f.write("\nGap_Closure\n")
    metrics["gap_closure"].to_csv(f, index=False)

    f.write("\nMedian\n")
    f.write("AS\n")
    metrics["median_scores"].to_csv(f, index=False)
    f.write("\nVBS\n")
    metrics["median_vbs"].to_csv(f, index=False)
    f.write("\nSBS\n")
    metrics["median_sbs"].to_csv(f, index=False)
    f.write("\nGap_Closure\n")
    metrics["median_gap_closure"].to_csv(f, index=False)

    f.write("\nAccuracies\n")
    metrics["accuracies"].to_csv(f, index=False)
    f.write("\nF1\n")
    metrics["f1"].to_csv(f, index=False)
    f.write("\nPick_Rate\n")
    metrics["pick_rate"].to_csv(f, header=["rate"])
    f.write("\nVBS_Pick_Rate\n")
    metrics["vbs_pick_rate"].to_csv(f, header=["rate"])

print("Wrote:", out_path)

Wrote: results/bbob/LPO/res_16_k_views_16.csv
